# Accessing DP2 metadetection shear catalogs

Contact author: Alex Broughton
<br>Date: 


In [ ]:
! eups list -s | grep lsst_distrib

## Introduction

This notebook shows how to load **per-patch** metadetection shear tables from the Rubin Butler, inspect configuration (which bands enter detection vs photometry), and view the matching **cell-based coadd** images (science, noise, PSF mosaic) for one tract/patch.

**Datasets**

| Dataset | Use case |
|--------|----------|
| `object_shear_patch` | One patch at a time; good for overlays on images and patch-local checks. |
| `object_shear_all` | Many patches concatenated; convenient for survey-level diagnostics (query refs first; tables can be large). |

Paths and the **collection** string below match the DP2 prep repo layout on SDF; adjust if your environment uses a different repo tag.

- Load per-patch metadetection shear tables from the Rubin Butler and inspect metadetect configuration (detection vs photometry bands).
- Display cell-based coadd images (science, noise, PSF mosaic) for a tract/patch example.
- Compare patch-level `object_shear_patch` with survey-level `object_shear_all` references.


## 1.0 Set Up


In [ ]:
### Import packages + setup common variables and useful functions
# NumPy: array handling for quick plots (e.g. PSF mosaic).
import numpy as np
import matplotlib.pyplot as plt

from lsst.daf.butler import Butler

# DP2 cell-based skymap and instrument; collection must match your installed DRP outputs.
REPO = "/sdf/data/rubin/repo/dp2_prep"
COLLECTION = "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3"

butler = Butler(
    REPO,
    collections=[COLLECTION],
    skymap="lsst_cells_v2",
    instrument="LSSTCam",
)

config_refs = butler.query_datasets("metadetectionShear_config")
metadetect_config = butler.get(config_refs[0])

## 2.0 Butler and metadetection config

Construct a `Butler` pointed at the DP2 data repository, then load `metadetectionShear_config`. That config object documents which bands are used for metadetect shear vs photometry (same schema idea applies across patches).


### Bands in metadetect shear

Which bands are combined for detection and shear measurement?

In [ ]:
metadetect_config.metadetect.shear_bands

### Photometry bands

Which bands have photometry measured (may differ from shear/detection bands)?

In [ ]:
metadetect_config.photometry_bands

## 3.0 Per-patch catalog: `object_shear_patch`

Pick a tract and patch (`dataId`). The values below are one valid example for this collection; to discover others, query the registry (see the end of the notebook) or use tooling that lists tract/patch coverage.

In [ ]:
# Minimal dataId for patch-level products (skymap and instrument already set on the Butler).
dataId = {
    "tract": 9813,
    "patch": 42,
}

In [ ]:
# PyArrow table (efficient columnar storage); prefer Arrow/Dask APIs for large analyses.
object_shear_patch = butler.get("object_shear_patch", dataId=dataId)
object_shear_patch

### Quick look with pandas (optional)

Converting the full table to pandas loads everything into memory. For a **quick interactive peek** at rows and dtypes, `to_pandas()` is fine; for science pipelines, stay in Arrow or use column projections.

In [ ]:
object_shear_patch.to_pandas().head()

### Column names and descriptions

The Arrow **schema** is shared across these catalogs: field names and short descriptions without pulling row data.

In [ ]:
object_shear_patch.schema

## 4.0 Cell coadd for the same patch

Load the **predetection** cell coadd (`deep_coadd_cell_predetection`), stitch cells into one image, and draw cell boundaries. We use band `i` here; change `band` to match your science goal.

Display uses **`lsst.afw.display`** (`Display`), the standard stack viewer; `matplotlib` is a simple default backend.

In [ ]:
band = "i"
multiple_cell_coadd = butler.get("deep_coadd_cell_predetection", band=band, **dataId)

# Stitch per-cell images into one large masked image; edges help when overlaying detections.
stitched_coadd = multiple_cell_coadd.stitch()
stitched_coadd.set_cell_edges()
exposure = stitched_coadd.asExposure()

In [ ]:
# Pick one backend for matplotlib figures in this notebook.
%matplotlib inline
# %matplotlib widget  # optional: interactive zoom/pan (can be finicky)

from lsst.afw.display import Display

afw_display_backend = "matplotlib"  # or "firefly" where configured

disp = Display(frame=1, backend=afw_display_backend)
disp.scale("linear", "zscale")
disp.image(exposure)
disp.show()

### Noise image

The stitched coadd also exposes a **noise** realisation (same geometry as the science image).

In [ ]:
noise_image = stitched_coadd.asMaskedImage(noise_index=0)

disp = Display(frame=2, backend=afw_display_backend)
disp.scale("linear", "zscale")
disp.image(noise_image)
disp.show()

### PSF mosaic at cell centers

`explode().psf_image` builds a mosaic of PSFs at cell centers for quick inspection. This pattern is convenient for DP2-era outputs; if stack APIs evolve, prefer documented PSF access for long-lived code.

In [ ]:
psf_mosaic = multiple_cell_coadd.explode().psf_image
psf_mosaic.array.shape  # (height, width) of mosaic

In [ ]:
disp = Display(frame=3, backend=afw_display_backend)
disp.scale("linear", "zscale")
disp.image(psf_mosaic)
disp.show()

Same mosaic with plain matplotlib (WCS not applied; fine for a sanity check).

In [ ]:
plt.figure(figsize=(8, 8))
plt.imshow(
    psf_mosaic.array,
    origin="lower",
    interpolation="none",
    cmap="viridis",
)
plt.colorbar(label="PSF mosaic (arbitrary units)")
plt.title(f"PSF mosaic — tract {dataId['tract']} patch {dataId['patch']} band {band}")
plt.xlabel("pixel x")
plt.ylabel("pixel y")
plt.tight_layout()

## 5.0 Survey-level catalog: `object_shear_all`

`object_shear_all` concatenates many per-patch tables for run-level analysis. **Do not** blindly `butler.get` the full dataset in a notebook unless you know it fits in memory.

Start by listing **dataset references** (`query_datasets`): each ref carries a `dataId` you can use to understand coverage or to load subsets with the registry/quantum graph tools appropriate to your workflow.

In [ ]:
object_shear_all_refs = list(butler.query_datasets("object_shear_all"))
len(object_shear_all_refs)

# Example: inspect one ref (do not confuse with metadetection *config* refs).
example_ref = object_shear_all_refs[0]
example_ref.datasetType.name, example_ref.dataId